In [1]:
# Calculate OSDMA8 population weighted exposure

In [2]:
import xarray as xr
import os
import numpy as np
from utils.processing_utils import weighted_sum

In [ ]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [ ]:
pop_ssp2 = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_2000-2100.nc")
country_mask = xr.open_dataarray(f"{BMR_DIR}GBD_Country_Masks_0.10_popgrid_newlabels.nc")

# Interpolate population across decades
new_year = range(2000, 2101)
new_pop = pop_ssp2.interp(year=new_year)

In [4]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/exposure/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        # Load data array
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3 = xr.open_dataarray(f"{O3_DIR}OSDMA8_BC_popgrid_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")
        population = new_pop.sel(year=o3.year)  # Select same years as o3 data for population

        weighted_value = population * o3

        country_list = []

        for i in range(204):
            mask = country_mask.isel(country=i)
            country_weight = weighted_sum(xr.where(mask == 1, weighted_value, np.nan))
            pop_country = weighted_sum(xr.where(mask == 1, population, np.nan))
            country_pop_weighted = country_weight / pop_country
            country_list.append(country_pop_weighted)

        pop_weighted_exposure = xr.concat(country_list, "country")

        out_file = f"OSDMA8_country_population_weighted_exposure_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        pop_weighted_exposure.to_netcdf(out_path)

Processing SSP245, Ensemble 06
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_06_2020-2069.nc
Processing SSP245, Ensemble 07
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_07_2020-2069.nc
Processing SSP245, Ensemble 08
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_08_2020-2069.nc
Processing SSP245, Ensemble 09
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_09_2020-2069.nc
Processing SSP245, Ensemble 10
Saving to /glade/work/awells/air_quality/CESM/ozone/exposure/OSDMA8_country_population_weighted_exposure_CESM2_SSP245_10_2020-2069.nc
